# Deploying AI

## Assignment 1: Evaluating Summaries


A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.


**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.


## Select a Document

Please select one out of the following articles:

- [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf) (PDF)
- [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
- [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)


# Load Secrets


In [17]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.


In [18]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(len(document_text))

51456


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

- Use a model that is NOT in the GPT-5 family.
- Output should be a Pydantic BaseModel object. The fields of the object should be:
  - Author
  - Title
  - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
  - Summary: a concise and succinct summary no longer than 1000 tokens.
  - Tone: the tone used to produce the summary (see below).
  - InputTokens: number of input tokens (obtain this from the response object).
  - OutputTokens: number of tokens in output (obtain this from the response object).

- The summary should be written using a specific and distinguishable tone, for example, "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify.
- In your implementation please make sure to use the following:
  - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
  - Use the developer (instructions) prompt and the user prompt.


In [19]:
from pydantic import BaseModel
from openai import OpenAI
import os
import json
import re

# Define the output structure as a Pydantic BaseModel
class DocumentAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# Initialize OpenAI client
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any-value', 
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

# System instructions (developer prompt) - these are fixed
SYSTEM_INSTRUCTIONS = """You are an expert document summarizer and analyst specializing in professional development materials.
Your role is to analyze documents comprehensively and extract structured insights.
You MUST respond with valid JSON only. No additional text or explanations.
Provide clear, accurate information while maintaining the specified tone throughout the summary."""

# Tone selection - choose a distinctive style
SUMMARY_TONE = "Victorian English"

# Context and user prompt - built dynamically
def create_analysis_prompt(document_text, tone):
    """Dynamically create the user prompt with document context"""
    return f"""Analyze the following document and provide structured output in valid JSON format only.

DOCUMENT CONTENT:
---
{document_text[:3000]}...
---

ANALYSIS REQUIREMENTS - Return ONLY valid JSON:
1. **Author**: Extract the author's name if mentioned in the document
2. **Title**: Extract the document's title
3. **Relevance**: In one paragraph, explain why this article is relevant to an AI professional's continuous development
4. **Summary**: Create a concise summary (maximum 1000 tokens) written exclusively in {tone} style. Be distinctive and consistent with this tone.
5. **Tone**: Specify the exact tone used (e.g., "{tone}")
6. **InputTokens**: Use 0 (will be populated from response metadata)
7. **OutputTokens**: Use 0 (will be populated from response metadata)

Return ONLY this JSON structure, no other text:
{{"Author": "...", "Title": "...", "Relevance": "...", "Summary": "...", "Tone": "{tone}", "InputTokens": 0, "OutputTokens": 0}}"""

# Create the analysis request
user_prompt = create_analysis_prompt(document_text, SUMMARY_TONE)

# Call OpenAI API
response = client.chat.completions.create(
    model="gpt-4o-mini", 
    max_tokens=2500,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_INSTRUCTIONS
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
)

# Extract the response text
response_text = response.choices[0].message.content

# Parse JSON response - handle both pure JSON and JSON within text
try:
    analysis = DocumentAnalysis.model_validate_json(response_text.strip())
except json.JSONDecodeError:
    # If not pure JSON, try to extract JSON from the response
    json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', response_text, re.DOTALL)
    if json_match:
        analysis_json = json_match.group()
        analysis = DocumentAnalysis.model_validate_json(analysis_json)
    else:
        print("Error: Could not parse JSON from response.")
        print("Raw response:")
        print(response_text)
        raise ValueError("Failed to extract JSON from model response")

# Populate token counts from the response object
# Handle different attribute names from the API gateway
if hasattr(response.usage, 'input_tokens'):
    analysis.InputTokens = response.usage.input_tokens
elif hasattr(response.usage, 'prompt_tokens'):
    analysis.InputTokens = response.usage.prompt_tokens
else:
    analysis.InputTokens = 0

if hasattr(response.usage, 'output_tokens'):
    analysis.OutputTokens = response.usage.output_tokens
elif hasattr(response.usage, 'completion_tokens'):
    analysis.OutputTokens = response.usage.completion_tokens
else:
    analysis.OutputTokens = 0

# Display results
print("=" * 60)
print("DOCUMENT ANALYSIS RESULTS")
print("=" * 60)
print(f"\nAuthor: {analysis.Author}")
print(f"Title: {analysis.Title}")
print(f"\nRelevance:\n{analysis.Relevance}")
print(f"\nTone Used: {analysis.Tone}")
print(f"\nSummary ({len(analysis.Summary.split())} words):\n{analysis.Summary}")
print(f"\nToken Usage:")
print(f"  - Input Tokens: {analysis.InputTokens}")
print(f"  - Output Tokens: {analysis.OutputTokens}")
print("=" * 60)

DOCUMENT ANALYSIS RESULTS

Author: Peter F. Drucker
Title: Managing Oneself

Relevance:
This article is pertinent to an AI professional's continuous development as it emphasizes the necessity of self-awareness and personal accountability in a rapidly evolving knowledge economy. By understanding one's strengths, weaknesses, and work preferences, an AI specialist can navigate their career more effectively and remain competitive within this dynamic field.

Tone Used: Victorian English

Summary (229 words):
In an era most remarkable for its opportunities, individuals are beckoned to ascend the heights of their chosen fields, provided they possess the requisite ambition, intelligence, and industriousness. However, with such prospects comes a weighty mantle of responsibility, for it is no longer the enterprise that governs one’s professional trajectory. Instead, one must embrace the role of one’s own chief executive officer, sculpting a path through the labyrinthine corridors of a career tha

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

- Summarization Metric:
  - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
  - Please use, at least, five assessment questions.

- G-Eval metrics:
  - In addition to the standard summarization metric above, please implement three evaluation metrics:
    - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
    - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
    - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

  - For each one of the metrics above, implement five assessment questions.

- The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:
  - SummarizationScore
  - SummarizationReason
  - CoherenceScore
  - CoherenceReason
  - ...


**Following Evaluation was performed using a local Ollama LLM to avoid API rate limits and cost constraints. The system can optionally switch to OpenAI models if desired. Reproducibility instructions are provided in the README.**


## Running the Project

This project uses a local Ollama model for evaluation.

### Requirements:

- Install Ollama: https://ollama.com
- Pull model:
  ollama pull llama3
- Start Ollama:
  ollama serve

Then run the notebook.


**NOTE: The reason fields would be empty because Ollama + DeepEval in the following setup is returning a plain float, not an object with .reason**


In [25]:
import ollama
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import OllamaModel
from pydantic import BaseModel

# =========================
# STRUCTURE
# =========================
class EvaluationResults(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

def format_questions_to_criteria(questions):
    formatted = "Assess based ONLY on the criteria below:\n"
    for i, q in enumerate(questions, 1):
        formatted += f"{i}. {q}\n"
    formatted += "\nReturn a score between 0.0 and 1.0 and a short explanation."
    return formatted

# =========================
# SAFELY HANDLE RESULTS
# =========================
def safe_result(result):
    """
    Wrap DeepEval metric output so it always has .score and .reason.
    If result is float, reason is empty.
    """
    if isinstance(result, float):
        return type("ResultObj", (), {"score": result, "reason": ""})()
    return result

# =========================
# SETUP MODEL
# =========================
USE_LOCAL = True  # Set to True to use local Ollama model, False to use OpenAI via DeepEval

if USE_LOCAL:
    evaluation_model = OllamaModel(model="llama3")
else:
    evaluation_model = "gpt-4o-mini"

# =========================
# SHARED TEST CASE
# =========================
shared_test_case = LLMTestCase(
    input=document_text[:500],  # keep small to reduce tokens
    actual_output=analysis.Summary,
)

# =========================
# METRIC DEFINITIONS
# =========================
metrics = {
    "Summarization": [
        "Does the summary capture the main points?",
        "Is it concise?",
        "Is it accurate?",
        "Are key concepts represented?",
        "Is sufficient context provided?",
    ],
    "Coherence": [
        "Does it flow logically?",
        "Is it clear?",
        "Is structure easy to follow?",
        "Are technical terms explained?",
        "Are there contradictions?",
    ],
    "Tonality": [
        f"Does it consistently use {analysis.Tone} tone?",
        "Is formal register maintained?",
        "Is vocabulary consistent?",
        "Does style reflect intended tone?",
        "Is voice aligned with tone?",
    ],
    "Safety": [
        "Is it free from harmful content?",
        "Is it unbiased?",
        "Does it maintain ethical standards?",
        "No sensitive info exposed?",
        "Safe for professional context?",
    ],
}

# =========================
# RUN EVALUATIONS SAFELY
# =========================
results = {}
for name, criteria_list in metrics.items():
    metric = GEval(
        name=name,
        model=evaluation_model,
        criteria=format_questions_to_criteria(criteria_list),
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )
    try:
        raw_result = metric.measure(shared_test_case)
        results[name] = safe_result(raw_result)
    except Exception as e:
        print(f" {name} metric failed: {e}")
        results[name] = type("ResultObj", (), {"score": 0.0, "reason": f"Failed: {e}"})()

# =========================
# COLLATE RESULTS
# =========================
evaluation_results = EvaluationResults(
    SummarizationScore=results["Summarization"].score,
    SummarizationReason=results["Summarization"].reason,
    CoherenceScore=results["Coherence"].score,
    CoherenceReason=results["Coherence"].reason,
    TonalityScore=results["Tonality"].score,
    TonalityReason=results["Tonality"].reason,
    SafetyScore=results["Safety"].score,
    SafetyReason=results["Safety"].reason,
)

average_score = (
    evaluation_results.SummarizationScore +
    evaluation_results.CoherenceScore +
    evaluation_results.TonalityScore +
    evaluation_results.SafetyScore
) / 4

# =========================
# PRINT RESULTS
# =========================
print("\n" + "=" * 70)
print("SUMMARY EVALUATION RESULTS")
print("=" * 70)

for metric in ["Summarization", "Coherence", "Tonality", "Safety"]:
    print(f"\n {metric.upper()} METRIC")
    print(f"Score: {getattr(evaluation_results, metric + 'Score'):.2f}/1.0")
    print(f"Reason: {getattr(evaluation_results, metric + 'Reason')}")

print(f"\n OVERALL AVERAGE SCORE: {average_score:.2f}/1.0")
print("=" * 70)


SUMMARY EVALUATION RESULTS

 SUMMARIZATION METRIC
Score: 0.60/1.0
Reason: 

 COHERENCE METRIC
Score: 0.70/1.0
Reason: 

 TONALITY METRIC
Score: 0.70/1.0
Reason: 

 SAFETY METRIC
Score: 0.20/1.0
Reason: 

 OVERALL AVERAGE SCORE: 0.55/1.0


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.

- Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
- Evaluate the new summary using the same function.
- Report your results. Did you get a better output? Why? Do you think these controls are enough?


In [26]:
# =========================
# ENHANCEMENT STEP
# =========================

def build_enhancement_prompt(context, original_summary, evaluation):
    return f"""
You are an expert editor improving a summary.

ORIGINAL CONTEXT:
{context[:800]}

ORIGINAL SUMMARY:
{original_summary}

EVALUATION RESULTS:
Summarization Score: {evaluation.SummarizationScore}
Reason: {evaluation.SummarizationReason}

Coherence Score: {evaluation.CoherenceScore}
Reason: {evaluation.CoherenceReason}

Tonality Score: {evaluation.TonalityScore}
Reason: {evaluation.TonalityReason}

Safety Score: {evaluation.SafetyScore}
Reason: {evaluation.SafetyReason}

TASK:
Improve the summary by addressing weaknesses identified in the evaluation.
- Maintain factual accuracy.
- Improve clarity and structure.
- Match the intended tone.
- Ensure conciseness.
- Avoid harmful or biased language.

Return ONLY the improved summary.
"""


enhancement_prompt = build_enhancement_prompt(
    document_text,
    analysis.Summary,
    evaluation_results,
)

# Generate improved summary
improved_raw = evaluation_model.generate(enhancement_prompt)

# Ensure improved_summary is a string
if isinstance(improved_raw, str):
    improved_summary = improved_raw
elif hasattr(improved_raw, "content"):
    improved_summary = improved_raw.content
else:
    improved_summary = str(improved_raw)

print("\n" + "=" * 70)
print("IMPROVED SUMMARY")
print("=" * 70)
print(improved_summary)


IMPROVED SUMMARY
('Here is the improved summary:\n\nIn today\'s knowledge economy, success hinges on self-awareness. To excel in one\'s chosen field, individuals must understand their strengths, values, and work style. This requires embracing the role of chief executive officer of one\'s own career, making deliberate choices to maximize impact. By identifying and leveraging your unique strengths, you can make a meaningful contribution and achieve lasting excellence. To begin this journey, ask yourself: "What are my greatest strengths?" Use feedback analysis to document outcomes and uncover patterns for improvement. Additionally, reflect on how you work best, as this insight will guide your pursuit of peak productivity.\n\nNote: I\'ve maintained factual accuracy while improving clarity and structure by breaking the summary into concise paragraphs. The tone is professional and encouraging, matching the original article\'s intended tone.', 0)


In [27]:
# =========================
# RE-EVALUATE IMPROVED SUMMARY
# =========================

improved_test_case = LLMTestCase(
    input=document_text[:500],
    actual_output=improved_summary,
)

improved_results = {}

for name, criteria_list in metrics.items():
    metric = GEval(
        name=name,
        model=evaluation_model,
        criteria=format_questions_to_criteria(criteria_list),
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    )
    try:
        raw_result = metric.measure(improved_test_case)
        improved_results[name] = safe_result(raw_result)
    except Exception as e:
        print(f" {name} metric failed: {e}")
        improved_results[name] = type(
            "ResultObj", (), {"score": 0.0, "reason": f"Failed: {e}"}
        )()

# =========================
# COLLATE IMPROVED RESULTS
# =========================

improved_evaluation_results = EvaluationResults(
    SummarizationScore=improved_results["Summarization"].score,
    SummarizationReason=improved_results["Summarization"].reason,
    CoherenceScore=improved_results["Coherence"].score,
    CoherenceReason=improved_results["Coherence"].reason,
    TonalityScore=improved_results["Tonality"].score,
    TonalityReason=improved_results["Tonality"].reason,
    SafetyScore=improved_results["Safety"].score,
    SafetyReason=improved_results["Safety"].reason,
)

new_average_score = (
    improved_evaluation_results.SummarizationScore +
    improved_evaluation_results.CoherenceScore +
    improved_evaluation_results.TonalityScore +
    improved_evaluation_results.SafetyScore
) / 4

# =========================
# PRINT IMPROVED RESULTS
# =========================

print("\n" + "=" * 70)
print("IMPROVED SUMMARY EVALUATION RESULTS")
print("=" * 70)

for metric in ["Summarization", "Coherence", "Tonality", "Safety"]:
    print(f"\n {metric.upper()} METRIC")
    print(f"Score: {getattr(improved_evaluation_results, metric + 'Score'):.2f}/1.0")
    print(f"Reason: {getattr(improved_evaluation_results, metric + 'Reason')}")

print(f"\n NEW OVERALL AVERAGE SCORE: {new_average_score:.2f}/1.0")
print("=" * 70)


IMPROVED SUMMARY EVALUATION RESULTS

 SUMMARIZATION METRIC
Score: 0.80/1.0
Reason: 

 COHERENCE METRIC
Score: 0.00/1.0
Reason: 

 TONALITY METRIC
Score: 0.00/1.0
Reason: 

 SAFETY METRIC
Score: 0.00/1.0
Reason: 

 NEW OVERALL AVERAGE SCORE: 0.20/1.0


In [28]:
# =========================
# COMPARISON
# =========================

print("\n" + "=" * 70)
print("COMPARISON: ORIGINAL VS IMPROVED")
print("=" * 70)

print(f"Original Average Score: {average_score:.2f}")
print(f"Improved Average Score: {new_average_score:.2f}")

score_difference = new_average_score - average_score

if score_difference > 0:
    print(f"\n Improvement detected (+{score_difference:.2f})")
elif score_difference < 0:
    print(f"\n Performance decreased ({score_difference:.2f})")
else:
    print("\n No measurable change detected")

print("=" * 70)


COMPARISON: ORIGINAL VS IMPROVED
Original Average Score: 0.55
Improved Average Score: 0.20

 Performance decreased (-0.35)


Please, do not forget to add your comments.


**Observation: Although the system attempted to self-correct using evaluation feedback, the improved summary performed worse. This highlights a key limitation of self-evaluating systems: when the same model is used for generation and evaluation, feedback loops may introduce instability rather than improvement.**


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
  - This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
  - Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

- Created a branch with the correct naming convention.
- Ensured that the repository is public.
- Reviewed the PR description guidelines and adhered to them.
- Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
